# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, with all dataset entities referenced by their `@id` fields for full reproducibility and clarity.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name')}: {getattr(metadata, 'description')}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets, their @id, and fields, all by @id.

record_sets = list(dataset.metadata.record_sets)
print("Available Record Sets:")
for rs in record_sets:
    print(f"- Record Set name: {getattr(rs, 'name', None)} | @id: {getattr(rs, '@id', None)}")
    if hasattr(rs, 'fields'):
        print("  fields (by @id):")
        for f in rs.fields:
            print(f"    - {getattr(f, 'name', None)} (@id={getattr(f, '@id', None)}) | dataType: {getattr(f, 'data_type', None)}")
print("\nExample records from the first record set:")
# Print only first few records for demonstration (substitute with exact @id in later cells).
if record_sets:
    rs_id = getattr(record_sets[0], '@id')
    for i, rec in enumerate(dataset.records(record_set=rs_id)):
        if i >= 3:
            break
        print(rec)


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set by its `@id`
dataframes = {}
record_set_ids = [getattr(rs, '@id') for rs in record_sets]

for rs in record_sets:
    rs_id = getattr(rs, '@id')
    print(f"Loading records for record_set @id: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"    Columns: {df.columns.tolist()}")
        print(f"    Number of records: {len(df)}\n")
    except Exception as e:
        print(f"    Error: {e}\n")

# Choose first available record set for demonstration
example_record_set_id = record_set_ids[0] if record_set_ids else None
if example_record_set_id:
    print(f"Columns in record set {example_record_set_id}:")
    print(dataframes[example_record_set_id].columns.tolist())
    dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
This section works by referencing all fields and record sets by their `@id`s.

In [ ]:
# Select a record set and a numeric field for EDA, using their @id

# Find a numeric field from the first record set for demonstration
numeric_field_id = None
group_field_id = None
rs = record_sets[0] if record_sets else None
if rs is not None and hasattr(rs, 'fields'):
    for f in rs.fields:
        dt = getattr(f, 'data_type', None)
        # Use the first Integer or Float field
        if dt in ('Float', 'Integer', 'Number') and numeric_field_id is None:
            numeric_field_id = getattr(f, '@id')
        # Use the first non-numeric field as group field
        if dt in ('Text', 'String') and group_field_id is None:
            group_field_id = getattr(f, '@id')

print(f'Using record_set @id: {example_record_set_id}')
print(f'Numeric field for EDA: {numeric_field_id}')
print(f'Group field: {group_field_id}')

edf = dataframes[example_record_set_id].copy()
threshold = edf[numeric_field_id].mean() if numeric_field_id and not edf.empty else 0
# Filter records with value greater than the mean
if numeric_field_id:
    filtered_df = edf[edf[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) 
        / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field if exists
    if group_field_id and group_field_id in edf.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize distributions of the selected numeric field, and relationships with group category if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and not edf.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(edf[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

if group_field_id and numeric_field_id and group_field_id in edf.columns:
    plt.figure(figsize=(12, 4))
    sns.boxplot(data=edf, x=group_field_id, y=numeric_field_id)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR<sup>2</sup> dataset "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" using the `mlcroissant` library. 

- All explorations referenced record sets, fields, and columns using their unique `@id` values for reproducibility and documentation.
- We've summarized available record sets and demonstrated filtering and normalization of a numeric field, as well as visualizing its distribution and group-level trends.

This workflow can be extended for deeper domain-specific analyses, including advanced statistical or machine learning tasks, leveraging the dataset's rich structure and Croissant-based provenance traceability.